This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## ConvNet architecture patterns

### Modularity, hierarchy, and reuse

### Residual connections

In [ ]:
import torch
from torch import nn

# PyTorch: there's no Functional API. We express the residual connection as an
# nn.Module whose forward() adds the skip branch. Conv2d works on NCHW tensors,
# so the example input is (batch, 3, 32, 32) instead of Keras's NHWC.
class ResidualExample(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        # PyTorch: padding="same" keeps the spatial size for the 3x3 conv.
        self.conv2 = nn.Conv2d(32, 64, 3, padding="same")
        # 1x1 conv projects the residual to the matching channel count (32 -> 64).
        self.residual = nn.Conv2d(32, 64, 1)

    def forward(self, inputs):
        x = torch.relu(self.conv1(inputs))
        residual = x
        x = torch.relu(self.conv2(x))
        residual = self.residual(residual)
        x = x + residual
        return x

In [ ]:
# PyTorch: same idea with downsampling. MaxPool2d(2) halves the spatial size, so
# the residual branch uses a stride-2 1x1 conv to match (and "same" padding on the
# pool keeps the dimensions aligned for the add).
class ResidualWithPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3, padding="same")
        self.pool = nn.MaxPool2d(2, padding=0, ceil_mode=True)
        self.residual = nn.Conv2d(32, 64, 1, stride=2)

    def forward(self, inputs):
        x = torch.relu(self.conv1(inputs))
        residual = x
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        residual = self.residual(residual)
        x = x + residual
        return x

In [ ]:
# PyTorch: the residual_block is a small nn.Module. Because PyTorch needs explicit
# channel counts, we pass in_filters as well. Rescaling becomes a plain divide by
# 255 in forward(), and GlobalAveragePooling2D is x.mean over the spatial dims.
class ResidualBlock(nn.Module):
    def __init__(self, in_filters, filters, pooling=False):
        super().__init__()
        self.pooling = pooling
        self.conv1 = nn.Conv2d(in_filters, filters, 3, padding="same")
        self.conv2 = nn.Conv2d(filters, filters, 3, padding="same")
        if pooling:
            self.pool = nn.MaxPool2d(2, padding=0, ceil_mode=True)
            self.residual = nn.Conv2d(in_filters, filters, 1, stride=2)
        elif filters != in_filters:
            self.residual = nn.Conv2d(in_filters, filters, 1)
        else:
            self.residual = None

    def forward(self, x):
        residual = x
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        if self.pooling:
            x = self.pool(x)
            residual = self.residual(residual)
        elif self.residual is not None:
            residual = self.residual(residual)
        x = x + residual
        return x

class ResidualModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ResidualBlock(3, filters=32, pooling=True)
        self.block2 = ResidualBlock(32, filters=64, pooling=True)
        self.block3 = ResidualBlock(64, filters=128, pooling=False)
        self.classifier = nn.Linear(128, 1)

    def forward(self, inputs):
        x = inputs / 255.0
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.mean(dim=(2, 3))  # GlobalAveragePooling2D
        # PyTorch: drop the sigmoid; use BCEWithLogitsLoss on these logits.
        outputs = self.classifier(x)
        return outputs

model = ResidualModel()

### Batch normalization

### Depthwise separable convolutions

### Putting it together: A mini Xception-like model

In [0]:
import kagglehub

kagglehub.login()

In [0]:
import zipfile

download_path = kagglehub.competition_download("dogs-vs-cats")

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [ ]:
import os, shutil, pathlib
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

batch_size = 64
image_size = (180, 180)
# PyTorch: image_dataset_from_directory becomes ImageFolder + DataLoader. The
# transform resizes to image_size and ToTensor yields NCHW float tensors in [0, 1]
# (so we skip the /255 rescaling that the model used on raw pixels above).
base_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
])
train_dataset = ImageFolder(new_base_dir / "train", transform=base_transform)
validation_dataset = ImageFolder(new_base_dir / "validation", transform=base_transform)
test_dataset = ImageFolder(new_base_dir / "test", transform=base_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# PyTorch: the Keras RandomFlip/RandomRotation/RandomZoom augmentation layers map
# to torchvision transforms. We build an augmented training DataLoader by giving
# the training ImageFolder a transform that adds the random ops before ToTensor.
augmentation_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=36),       # ~0.1 * 360 degrees
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),  # ~RandomZoom(0.2)
    transforms.ToTensor(),
])
augmented_train_dataset = ImageFolder(
    new_base_dir / "train", transform=augmentation_transform
)
augmented_train_loader = DataLoader(
    augmented_train_dataset, batch_size=batch_size, shuffle=True
)

In [ ]:
# PyTorch: SeparableConv2D = a depthwise Conv2d (groups=in_channels) followed by a
# 1x1 pointwise Conv2d. We build the mini-Xception model as an nn.Module. Inputs
# are NCHW and already in [0, 1] from ToTensor, so there's no Rescaling layer.
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding="same", bias=False):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            padding=padding, groups=in_channels, bias=bias,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=bias)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class XceptionBlock(nn.Module):
    def __init__(self, in_channels, size):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.sepconv1 = SeparableConv2d(in_channels, size, 3, padding="same", bias=False)
        self.bn2 = nn.BatchNorm2d(size)
        self.sepconv2 = SeparableConv2d(size, size, 3, padding="same", bias=False)
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)
        self.residual = nn.Conv2d(in_channels, size, 1, stride=2, bias=False)

    def forward(self, x):
        residual = x
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.sepconv1(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.sepconv2(x)
        x = self.pool(x)
        residual = self.residual(residual)
        x = x + residual
        return x

class MiniXception(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Conv2d(3, 32, kernel_size=5, bias=False)
        self.blocks = nn.ModuleList()
        in_channels = 32
        for size in [32, 64, 128, 256, 512]:
            self.blocks.append(XceptionBlock(in_channels, size))
            in_channels = size
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(512, 1)

    def forward(self, inputs):
        x = self.stem(inputs)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=(2, 3))  # GlobalAveragePooling2D
        x = self.dropout(x)
        # PyTorch: drop the sigmoid; BCEWithLogitsLoss expects raw logits.
        outputs = self.classifier(x)
        return outputs

model = MiniXception()

In [ ]:
# PyTorch: compile() + fit() become an explicit training loop. binary_crossentropy
# -> BCEWithLogitsLoss (works on the logits we kept), "adam" -> torch.optim.Adam.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(100):
    model.train()
    for images, labels in augmented_train_loader:
        images = images.to(device)
        # BCEWithLogitsLoss wants float targets shaped like the (N, 1) logits.
        targets = labels.float().unsqueeze(1).to(device)
        predictions = model(images)
        loss = loss_fn(predictions, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # PyTorch: per-epoch validation pass (replaces validation_data= in fit()).
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            targets = labels.float().unsqueeze(1).to(device)
            predictions = model(images)
            predicted = (torch.sigmoid(predictions) > 0.5).float()
            val_correct += (predicted == targets).sum().item()
            val_total += targets.size(0)
    print(f"Epoch {epoch}: loss {loss.item():.4f}, val_acc {val_correct / val_total:.4f}")

### Beyond convolution: Vision Transformers